In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300

## Configuration

In [2]:
# Update these paths to match your setup
BASE_PATH = Path("/mnt/c/Research/textual-time-series-for-sepsis/new_analysis/sepsis-20/tta_analysis/matches")
OUTPUT_PATH = Path("/mnt/c/Research/textual-time-series-for-sepsis/new_analysis/sepsis-20/tta_analysis/matches")

# Pilot names in desired order (for colors and legend)
PILOT_ORDER = ['DSL33']

# Color palette (matching R code)
COLORS = ['#000000', '#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', '#D55E00', '#CC79A7', '#AA2200']
COLOR_MAP = {pilot: COLORS[i] for i, pilot in enumerate(PILOT_ORDER)}

# Threshold to highlight with larger point
BOLD_THRESHOLD = 0.1

# Thresholds to sweep
THRESHOLDS = [
    0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1,
    0.11, 0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18, 0.19, 0.2,
    0.25, 0.3, 0.35, 0.4, 0.45, 0.5
]

## Load Data

In [3]:
# Dictionary mapping pilot names to their best_matches files
pilot_files = {
    pilot: BASE_PATH / pilot / 'best_matches2026-02-26.csv'
    for pilot in PILOT_ORDER
}

# Display file paths
print("Loading files:")
for pilot, path in pilot_files.items():
    exists = "✓" if path.exists() else "✗"
    print(f"  {exists} {pilot}: {path}")

Loading files:
  ✓ DSL33: /mnt/c/Research/textual-time-series-for-sepsis/new_analysis/sepsis-20/tta_analysis/matches/DSL33/best_matches2026-02-26.csv


In [13]:
def load_all_pilots(pilot_files: Dict[str, Path]) -> pd.DataFrame:
    """Load all best_matches files and combine them."""
    dfs = {}
    for pilot_name, filepath in pilot_files.items():
        df = pd.read_csv(filepath)
        df['pilot_name'] = pilot_name
        dfs[pilot_name] = df
    return dfs

# Load data
dfs = load_all_pilots(pilot_files)

df = dfs['DSL33']

print(f"\nLoaded {len(df):,} total matches")
print(f"Pilots: {df['pilot_name'].nunique()}")
print(f"Cases: {df['common.bns'].nunique() if 'common.bns' in df.columns else 'N/A'}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
# df.head()
df


Loaded 3,784 total matches
Pilots: 1
Cases: 20

Columns: ['common.bns', 'files', 'pilot.files', 'v1', 'v2', 'error.rate', 'idx', 'match.idx', 'ref.time', 'pilot.time', 'known_pilot', 'pilot_name']

First few rows:


,common.bns,files,pilot.files,v1,v2,error.rate,idx,match.idx,ref.time,pilot.time,known_pilot,pilot_name
0,2023-03-03_11-58-20_annotations_158331_yes_str...,/mnt/c/Research/textual-time-series-for-sepsis...,/mnt/c/Research/textual-time-series-for-sepsis...,Admission,admitted to the hospital,0.065274,1,3.0,-1.27,0.0,1.0,DSL33
1,2023-03-03_11-58-20_annotations_158331_yes_str...,/mnt/c/Research/textual-time-series-for-sepsis...,/mnt/c/Research/textual-time-series-for-sepsis...,relieved,systemic lupus erythematosus in remission,0.149522,27,7.0,-26.04,-672.0,1.0,DSL33
2,2023-03-03_11-58-20_annotations_158331_yes_str...,/mnt/c/Research/textual-time-series-for-sepsis...,/mnt/c/Research/textual-time-series-for-sepsis...,sublingual nitroglycerin,Lasix 20 mg po qd,0.159227,28,61.0,-26.04,144.0,1.0,DSL33
3,2023-03-03_11-58-20_annotations_158331_yes_str...,/mnt/c/Research/textual-time-series-for-sepsis...,/mnt/c/Research/textual-time-series-for-sepsis...,transferred,transferred to floor,0.068543,30,43.0,-17.47,72.0,1.0,DSL33
4,2023-03-03_11-58-20_annotations_158331_yes_str...,/mnt/c/Research/textual-time-series-for-sepsis...,/mnt/c/Research/textual-time-series-for-sepsis...,Myocardial infarction,myocardial infarction x3,0.077959,32,5.0,-3842.83,-672.0,1.0,DSL33
...,...,...,...,...,...,...,...,...,...,...,...,...
3779,2024-01-12_11-16-46_annotations_105276,/mnt/c/Research/textual-time-series-for-sepsis...,/mnt/c/Research/textual-time-series-for-sepsis...,making,no home medications,0.189063,28,15.0,48.58,0.0,1.0,DSL33
3780,2024-01-12_11-16-46_annotations_105276,/mnt/c/Research/textual-time-series-for-sepsis...,/mnt/c/Research/textual-time-series-for-sepsis...,recovery,discharged to home,0.127962,29,44.0,48.58,120.0,1.0,DSL33
3781,2024-01-12_11-16-46_annotations_105276,/mnt/c/Research/textual-time-series-for-sepsis...,/mnt/c/Research/textual-time-series-for-sepsis...,unremarkable,washings negative for malignant cells,0.140220,30,8.0,48.58,-72.0,1.0,DSL33
3782,2024-01-12_11-16-46_annotations_105276,/mnt/c/Research/textual-time-series-for-sepsis...,/mnt/c/Research/textual-time-series-for-sepsis...,mass,male,0.140152,31,2.0,124.19,0.0,1.0,DSL33


## Metric Calculation Functions

In [14]:
dfs['DSL33']

,common.bns,files,pilot.files,v1,v2,error.rate,idx,match.idx,ref.time,pilot.time,known_pilot,pilot_name
0,2023-03-03_11-58-20_annotations_158331_yes_str...,/mnt/c/Research/textual-time-series-for-sepsis...,/mnt/c/Research/textual-time-series-for-sepsis...,Admission,admitted to the hospital,0.065274,1,3.0,-1.27,0.0,1.0,DSL33
1,2023-03-03_11-58-20_annotations_158331_yes_str...,/mnt/c/Research/textual-time-series-for-sepsis...,/mnt/c/Research/textual-time-series-for-sepsis...,relieved,systemic lupus erythematosus in remission,0.149522,27,7.0,-26.04,-672.0,1.0,DSL33
2,2023-03-03_11-58-20_annotations_158331_yes_str...,/mnt/c/Research/textual-time-series-for-sepsis...,/mnt/c/Research/textual-time-series-for-sepsis...,sublingual nitroglycerin,Lasix 20 mg po qd,0.159227,28,61.0,-26.04,144.0,1.0,DSL33
3,2023-03-03_11-58-20_annotations_158331_yes_str...,/mnt/c/Research/textual-time-series-for-sepsis...,/mnt/c/Research/textual-time-series-for-sepsis...,transferred,transferred to floor,0.068543,30,43.0,-17.47,72.0,1.0,DSL33
4,2023-03-03_11-58-20_annotations_158331_yes_str...,/mnt/c/Research/textual-time-series-for-sepsis...,/mnt/c/Research/textual-time-series-for-sepsis...,Myocardial infarction,myocardial infarction x3,0.077959,32,5.0,-3842.83,-672.0,1.0,DSL33
...,...,...,...,...,...,...,...,...,...,...,...,...
3779,2024-01-12_11-16-46_annotations_105276,/mnt/c/Research/textual-time-series-for-sepsis...,/mnt/c/Research/textual-time-series-for-sepsis...,making,no home medications,0.189063,28,15.0,48.58,0.0,1.0,DSL33
3780,2024-01-12_11-16-46_annotations_105276,/mnt/c/Research/textual-time-series-for-sepsis...,/mnt/c/Research/textual-time-series-for-sepsis...,recovery,discharged to home,0.127962,29,44.0,48.58,120.0,1.0,DSL33
3781,2024-01-12_11-16-46_annotations_105276,/mnt/c/Research/textual-time-series-for-sepsis...,/mnt/c/Research/textual-time-series-for-sepsis...,unremarkable,washings negative for malignant cells,0.140220,30,8.0,48.58,-72.0,1.0,DSL33
3782,2024-01-12_11-16-46_annotations_105276,/mnt/c/Research/textual-time-series-for-sepsis...,/mnt/c/Research/textual-time-series-for-sepsis...,mass,male,0.140152,31,2.0,124.19,0.0,1.0,DSL33


### Event Match Rate

In [15]:
def calculate_match_rate(df: pd.DataFrame, threshold: float) -> pd.DataFrame:
    """
    Calculate event match rate.
    Match rate = percentage of events where error.rate < threshold
    """
    df_clean = df.copy()
    df_clean['error.rate'] = df_clean['error.rate'].fillna(np.inf)
    
    # Use groupby and agg to calculate match rate without triggering deprecation warning
    results = df_clean.groupby('pilot_name').agg(
        match_rate=('error.rate', lambda x: (x < threshold).mean())
    ).reset_index()
    
    return results

In [16]:
print("\nMatch Rates at Threshold 0.1:")

for pilot_, df in dfs.items():
    result = calculate_match_rate(df, threshold=0.1)
    print(f"Pilot: {pilot_}, Match Rate: {result['match_rate'].iloc[0]:.3f}")


Match Rates at Threshold 0.1:
Pilot: DSL33, Match Rate: 0.372


In [17]:
known_df = df[df['known_pilot'] == 1]
unknown_df = df[df['known_pilot'] != 1]
unknown_df_0 = df[df['known_pilot'] == 0]

print(f"Full Pilot Matches:" f"{len(df)}")
print(f"Known Pilot Matches: {len(known_df)}")
print(f"Unknown Pilot Matches: {len(unknown_df)}")
print(f"Unknown Pilot Matches (known_pilot == 0): {len(unknown_df_0)}")

Full Pilot Matches:3784
Known Pilot Matches: 1920
Unknown Pilot Matches: 1864
Unknown Pilot Matches (known_pilot == 0): 37


In [18]:
print("\nMatch Rates at Threshold 0.1:")

match_rate = calculate_match_rate(df, threshold=0.1)['match_rate'].iloc[0]
print(f"Pilot: {pilot_}, Match Rate: {match_rate:.3f}")
match_rate_known = calculate_match_rate(known_df, threshold=0.1)['match_rate'].iloc[0]
print(f"Known Pilot Match Rate: {match_rate_known:.3f}")
match_rate_unknown = calculate_match_rate(unknown_df, threshold=0.1)['match_rate'].iloc[0]
print(f"Unknown Pilot Match Rate: {match_rate_unknown:.3f}")
match_rate_unknown_0 = calculate_match_rate(unknown_df_0, threshold=0.1)['match_rate'].iloc[0]
print(f"Unknown Pilot (known_pilot == 0) Match Rate: {match_rate_unknown_0:.3f}")


Match Rates at Threshold 0.1:
Pilot: DSL33, Match Rate: 0.372
Known Pilot Match Rate: 0.722
Unknown Pilot Match Rate: 0.011
Unknown Pilot (known_pilot == 0) Match Rate: 0.541


In [19]:
# check if known is NA then pilot.time and v2 is NA
na_known = df[df['known_pilot'].isna()]
print(f"Rows with known_pilot NA: {len(na_known)}")
print(na_known[['pilot.time', 'v2']].isna().sum())

Rows with known_pilot NA: 1827
pilot.time    1827
v2            1827
dtype: int64


### Concordance

In [20]:
import numpy as np
import pandas as pd

def r_style_cindex_counts(
    df_one_file: pd.DataFrame,
    ties_value: float = 0.5,
    nonzero_threshold: float = 1e-5,
    truth_col: str = "ref.time",
    pred_col: str = "pilot.time",
):
    """
    Exact analog of the R my_cindex() logic used in your `remove` branch:

      crossing(pred, truth) x crossing(pred2, truth2)
      filter(abs(truth - truth2) > nonzero_threshold)
      score = 1 if (truth-truth2)*(pred-pred2) > 0
              ties_value if > -nonzero_threshold
              0 otherwise

    IMPORTANT R-faithful filtering:
      - R filters out NA pred (time.pilot) BEFORE computing C-index.
      - R does NOT explicitly filter out NA truth (time); pairs involving NA drop out later.
    """

    truth = pd.to_numeric(df_one_file[truth_col], errors="coerce")
    pred  = pd.to_numeric(df_one_file[pred_col], errors="coerce")

    # Match R remove-branch: filter(!is.na(time.pilot))
    keep = ~pred.isna()
    truth = truth[keep]
    pred  = pred[keep]

    n = int(len(pred))
    if n == 0:
        return dict(cindex=np.nan, score_sum=0.0, denom=0, concordant=0, ties=0, n_rows_used=0)

    t = truth.to_numpy(dtype=float)  # may contain NaN (allowed, matches R behavior)
    p = pred.to_numpy(dtype=float)

    # Ordered pairs (i, j), like R's crossing()
    dt = t[:, None] - t[None, :]
    dp = p[:, None] - p[None, :]

    # filter(abs(truth-truth2) > nonzero_threshold)
    # Note: comparisons involving NaN become False, which matches dplyr filter dropping NAs.
    mask = np.abs(dt) > nonzero_threshold

    prod = dt * dp

    denom = int(mask.sum())
    if denom == 0:
        return dict(cindex=np.nan, score_sum=0.0, denom=0, concordant=0, ties=0, n_rows_used=n)

    concordant = int(((prod > 0) & mask).sum())
    ties = int(((prod <= 0) & (prod > -nonzero_threshold) & mask).sum())

    score_sum = concordant + ties_value * ties
    cindex = score_sum / denom

    return dict(
        cindex=float(cindex),
        score_sum=float(score_sum),
        denom=denom,
        concordant=concordant,
        ties=ties,
        n_rows_used=n,
    )


def per_file_table_like_r(
    match_df: pd.DataFrame,
    threshold: float = 0.1,
    ties_value: float = 0.5,
    nonzero_threshold: float = 1e-5,
    truth_col: str = "ref.time",
    pred_col: str = "pilot.time",
) -> pd.DataFrame:
    """
    Produce the per-file table: (pilot, file) -> cindex + numerator/denominator etc.

    Matches the R remove-branch behavior:
      df %>% filter(error.rate < threshold) %>% ... %>% my_cindex(pred=time.pilot, truth=time)
    """

    df = match_df.copy()

    # R filters error.rate < threshold BEFORE nesting / cindex
    df = df[df["error.rate"] < threshold]

    # Handle either naming convention
    pilot_col = "pilot.names" if "pilot.names" in df.columns else "pilot_name"
    file_col = "common.bns"

    rows = []
    for i, ((pilot, common_bns), g) in enumerate(df.groupby([pilot_col, file_col], sort=True), start=1):
        out = r_style_cindex_counts(
            g,
            ties_value=ties_value,
            nonzero_threshold=nonzero_threshold,
            truth_col=truth_col,
            pred_col=pred_col,
        )
        rows.append({
            "pilot.names": pilot,     # keep R-style output column name
            "common.bns": common_bns,
            **out
        })

    return pd.DataFrame(rows)

In [21]:
tbl = per_file_table_like_r(df, threshold=0.1)

assert np.all(np.abs(tbl['concordant'].values/tbl['denom'].values - tbl['cindex'].values < 1e-5)), "Concordant/denom should equal cindex"

tbl

,pilot.names,common.bns,cindex,score_sum,denom,concordant,ties,n_rows_used
0,DSL33,2023-03-03_11-58-20_annotations_158331_yes_str...,0.740489,1635.0,2208,1472,326,50
1,DSL33,2023-03-04_11-12-47_annotations_133706_yes_str...,0.654139,1201.0,1836,950,502,44
2,DSL33,2023-03-06_08-14-49_annotations_160019_yes_str...,0.841680,1042.0,1238,930,224,37
3,DSL33,2023-03-07_12-53-02_annotations_145785_yes_str...,0.643934,5297.0,8226,2970,4654,106
4,DSL33,2023-03-16_09-29-19_annotations_125310_yes_str...,0.686873,1643.0,2392,1276,734,50
5,DSL33,2023-03-18_11-45-18_annotations_164788_yes_str...,0.858907,9180.0,10688,8416,1528,107
6,DSL33,2023-03-27_13-14-33_annotations_163261_yes_str...,0.718254,362.0,504,326,72,24
7,DSL33,2023-03-28_09-36-53_annotations_152818_yes_str...,0.742619,4276.0,5758,3292,1968,80
8,DSL33,2023-03-30_10-05-43_annotations_178300_yes_str...,0.801718,2333.0,2910,1856,954,60
9,DSL33,2023-03-31_10-31-14_annotations_162755_yes_str...,0.687769,3835.0,5576,2802,2066,79


In [22]:
aa = df[(df['common.bns'] == tbl['common.bns'].values[0]) & (df['error.rate'] < 0.1)].reset_index(drop=True)
# make time int
# aa['ref.time'] = pd.to_numeric(aa['ref.time'], errors='coerce').round().astype('Int64')
# aa['pilot.time'] = pd.to_numeric(aa['pilot.time'], errors='coerce').round().astype('Int64')
# drops rows with na
# df[(df['common.bns'] == tbl['common.bns'].values[0]) & (df['error.rate'] < 0.1)].isna().sum()
assert aa["match.idx"].sort_values().values.shape == aa["match.idx"].unique().shape, "match.idx should be unique per row"
assert aa["idx"].sort_values().values.shape == aa["idx"].unique().shape, "reference idx should be unique per row"


import numpy as np

def denom_like_r(truth_int: np.ndarray) -> int:
    # truth_int is 1D array of ints (after filtering rows)
    n = truth_int.shape[0]
    _, counts = np.unique(truth_int, return_counts=True)
    return int(n*n - np.sum(counts*counts))

aa.head()

,common.bns,files,pilot.files,v1,v2,error.rate,idx,match.idx,ref.time,pilot.time,known_pilot,pilot_name
0,2023-03-03_11-58-20_annotations_158331_yes_str...,/mnt/c/Research/textual-time-series-for-sepsis...,/mnt/c/Research/textual-time-series-for-sepsis...,Admission,admitted to the hospital,6.527391e-02,1,3.0,-1.27,0.0,1.0,DSL33
1,2023-03-03_11-58-20_annotations_158331_yes_str...,/mnt/c/Research/textual-time-series-for-sepsis...,/mnt/c/Research/textual-time-series-for-sepsis...,transferred,transferred to floor,6.854256e-02,30,43.0,-17.47,72.0,1.0,DSL33
2,2023-03-03_11-58-20_annotations_158331_yes_str...,/mnt/c/Research/textual-time-series-for-sepsis...,/mnt/c/Research/textual-time-series-for-sepsis...,Myocardial infarction,myocardial infarction x3,7.795894e-02,32,5.0,-3842.83,-672.0,1.0,DSL33
3,2023-03-03_11-58-20_annotations_158331_yes_str...,/mnt/c/Research/textual-time-series-for-sepsis...,/mnt/c/Research/textual-time-series-for-sepsis...,Coronary artery bypass graft,coronary artery bypass graft,9.992007e-14,33,6.0,-3817.86,-672.0,1.0,DSL33
4,2023-03-03_11-58-20_annotations_158331_yes_str...,/mnt/c/Research/textual-time-series-for-sepsis...,/mnt/c/Research/textual-time-series-for-sepsis...,afebrile,afebrile,1.236788e-13,36,49.0,-0.99,96.0,1.0,DSL33


In [23]:
ref_file = pd.read_csv(aa['files'].unique()[0])
pil_file = pd.read_csv(aa['pilot.files'].unique()[0])

# go over every row in aa, get the idx, match.idx, ref.time, pilot.time, then find the corresponding row in ref_file and pil_file and print them out for comparison
for i, row in aa.iterrows():
    idx = int(row['idx'])
    match_idx = int(row['match.idx'])
    ref_time = row['ref.time']
    ref_event = row['v1']
    pilot_time = row['pilot.time']
    pilot_event = row['v2']

    ref_row = ref_file.iloc[idx-1]
    pil_row = pil_file.iloc[match_idx-1]

    print(f"Row {i}:")
    print(f"  idx: {idx}, match.idx: {match_idx}")
    print(f"  ref.time: {ref_time}, ref.event: {ref_event}")
    print(f"  ref.time: {ref_row['time']}, ref.event: {ref_row['event']}")
    print(f"  pilot.time: {int(pilot_time)}, pilot.event: {pilot_event}")
    print(f"  pilot.time: {int(pil_row['time'])}, pilot.event: {pil_row['event']}")
    print("-" * 40)
    assert ref_time == ref_row['time'], f"ref.time mismatch at row {i}"
    assert ref_event == ref_row['event'], f"ref.event mismatch at row {i}"
    assert pilot_time == pil_row['time'], f"pilot.time mismatch at row {i}"
    assert pilot_event == pil_row['event'], f"pilot.event mismatch at row {i}"

Row 0:
  idx: 1, match.idx: 3
  ref.time: -1.27, ref.event: Admission
  ref.time: -1.27, ref.event: Admission
  pilot.time: 0, pilot.event: admitted to the hospital
  pilot.time: 0, pilot.event: admitted to the hospital
----------------------------------------
Row 1:
  idx: 30, match.idx: 43
  ref.time: -17.47, ref.event: transferred
  ref.time: -17.47, ref.event: transferred
  pilot.time: 72, pilot.event: transferred to floor
  pilot.time: 72, pilot.event: transferred to floor
----------------------------------------
Row 2:
  idx: 32, match.idx: 5
  ref.time: -3842.83, ref.event: Myocardial infarction
  ref.time: -3842.83, ref.event: Myocardial infarction
  pilot.time: -672, pilot.event: myocardial infarction x3
  pilot.time: -672, pilot.event: myocardial infarction x3
----------------------------------------
Row 3:
  idx: 33, match.idx: 6
  ref.time: -3817.86, ref.event: Coronary artery bypass graft
  ref.time: -3817.86, ref.event: Coronary artery bypass graft
  pilot.time: -672, pil

In [24]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Iterable, List, Tuple, Optional
import numpy as np

@dataclass
class ConcordanceResult:
    cindex: float
    denom: int
    concordant: int
    discordant: int
    pilot_ties: int
    score_sum: float

def concordance_from_aligned_times(
    ref: Iterable[float],
    pil: Iterable[float],
    *,
    ties_value: float = 0.5,
    ordered_pairs: bool = True,
) -> ConcordanceResult:
    """
    Concordance between aligned ref and pil time arrays, using ref to define comparable pairs.

    Comparable pair: ref[i] != ref[j].
    Score:
      - concordant: (ref[i]-ref[j])*(pil[i]-pil[j]) > 0  -> 1
      - discordant: < 0                                  -> 0
      - pilot tie: == 0 (while ref differs)              -> ties_value

    ordered_pairs=True  -> count (i,j) and (j,i) separately (matches the 'ordered' convention).
    ordered_pairs=False -> count each unordered pair once (i<j).

    Returns counts + cindex.
    """
    ref = np.asarray(list(ref), dtype=float)
    pil = np.asarray(list(pil), dtype=float)
    if ref.shape != pil.shape:
        raise ValueError(f"ref and pil must have same length, got {len(ref)} vs {len(pil)}")

    n = ref.size
    if n == 0:
        return ConcordanceResult(np.nan, 0, 0, 0, 0, 0.0)

    # Pairwise diffs
    dref = ref[:, None] - ref[None, :]
    dpil = pil[:, None] - pil[None, :]

    # Comparable iff ref differs
    comparable = dref != 0

    if not ordered_pairs:
        # keep only i<j
        iu = np.triu_indices(n, k=1)
        comparable = comparable[iu]
        prod = (dref * dpil)[iu]
        # For unordered, dref != 0 is equivalent to comparable mask; apply after
        prod = prod[comparable]
    else:
        prod = (dref * dpil)[comparable]

    denom = int(prod.size)
    if denom == 0:
        return ConcordanceResult(np.nan, 0, 0, 0, 0, 0.0)

    concordant = int(np.sum(prod > 0))
    discordant = int(np.sum(prod < 0))
    pilot_ties = int(np.sum(prod == 0))  # only possible when pil ties, since ref differs by construction

    score_sum = concordant + ties_value * pilot_ties
    cindex = float(score_sum / denom)

    return ConcordanceResult(
        cindex=cindex,
        denom=denom,
        concordant=concordant,
        discordant=discordant,
        pilot_ties=pilot_ties,
        score_sum=float(score_sum),
    )

def comparable_pairs_indices(
    ref: Iterable[float],
    *,
    ordered_pairs: bool = False,
    max_pairs: Optional[int] = None,
) -> List[Tuple[int, int]]:
    """
    Return the list of comparable pairs (i,j) where ref[i] != ref[j].
    By default unordered (i<j). Set ordered_pairs=True to return all i!=j.
    """
    ref = np.asarray(list(ref), dtype=float)
    n = ref.size
    out: List[Tuple[int, int]] = []
    if ordered_pairs:
        for i in range(n):
            for j in range(n):
                if i != j and ref[i] != ref[j]:
                    out.append((i, j))
                    if max_pairs is not None and len(out) >= max_pairs:
                        return out
    else:
        for i in range(n):
            for j in range(i + 1, n):
                if ref[i] != ref[j]:
                    out.append((i, j))
                    if max_pairs is not None and len(out) >= max_pairs:
                        return out
    return out

# ---- example you gave ----
ref = aa['ref.time'].values
pil = aa['pilot.time'].values

res_ord = concordance_from_aligned_times(ref, pil, ordered_pairs=True)
res_un  = concordance_from_aligned_times(ref, pil, ordered_pairs=False)

print("ORDERED:", res_ord)
print("UNORDERED:", res_un)
print("Some comparable pairs (unordered):", comparable_pairs_indices(ref, ordered_pairs=False, max_pairs=20))

ORDERED: ConcordanceResult(cindex=0.7404891304347826, denom=2208, concordant=1472, discordant=410, pilot_ties=326, score_sum=1635.0)
UNORDERED: ConcordanceResult(cindex=0.7404891304347826, denom=1104, concordant=736, discordant=205, pilot_ties=163, score_sum=817.5)
Some comparable pairs (unordered): [(0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (0, 8), (0, 9), (0, 10), (0, 11), (0, 12), (0, 13), (0, 14), (0, 15), (0, 16), (0, 17), (0, 18), (0, 19), (0, 20)]


In [25]:
ref = aa['ref.time'].values
pil = aa['pilot.time'].values

# ref = df_debugging_r['time'].values
# pil = df_debugging_r['time.pilot'].values

concordant_pair = []
disconcordant_pair = []
tie_pair = []
tie_uncomparable = []

for i in range(len(ref)):
    for j in range(i + 1, len(ref)):
        if ref[i] != ref[j]:
            if (ref[i] > ref[j]) and (pil[i] > pil[j]):
                concordant_pair.append((i, j))
            elif (ref[i] < ref[j]) and (pil[i] < pil[j]):
                concordant_pair.append((i, j))
            elif (ref[i] > ref[j]) and (pil[i] < pil[j]):
                disconcordant_pair.append((i, j))
            elif (ref[i] < ref[j]) and (pil[i] > pil[j]):
                disconcordant_pair.append((i, j))
            else:
                tie_pair.append((i, j))
        else:
            tie_uncomparable.append((i, j))

print("Total Pairs:" + str(len(ref) * (len(ref) - 1)//2))
print("Comparable Pairs (ref differs):", len(concordant_pair) + len(disconcordant_pair) + len(tie_pair))
print("Concordant pairs (ref and pil agree on order):", len(concordant_pair))
print("Discordant pairs (ref and pil disagree on order):", len(disconcordant_pair))
print("Pilot ties (ref differs but pil ties):", len(tie_pair))
print("Ref ties (ref ties, not comparable):", len(tie_uncomparable))
print("C-index:", (len(concordant_pair) + 0.5 * len(tie_pair)) / (len(concordant_pair) + len(disconcordant_pair) + len(tie_pair)))
        

Total Pairs:1225
Comparable Pairs (ref differs): 1104
Concordant pairs (ref and pil agree on order): 736
Discordant pairs (ref and pil disagree on order): 205
Pilot ties (ref differs but pil ties): 163
Ref ties (ref ties, not comparable): 121
C-index: 0.7404891304347826


In [27]:
import pandas as pd

dbg = pd.read_csv("concordance_input__DSL33__2023-03-03_11-58-20_annotations_158331_yes_structured.csv")

ref = dbg["time_num"].values
pil = dbg["time_pilot_num"].values

concordant_pair = []
disconcordant_pair = []
tie_pair = []
tie_uncomparable = []

eps = 1e-5

for i in range(len(ref)):
    for j in range(i + 1, len(ref)):
        if abs(ref[i] - ref[j]) > eps:
            prod = (ref[i] - ref[j]) * (pil[i] - pil[j])
            if prod > 0:
                concordant_pair.append((i, j))
            elif prod > -eps:
                tie_pair.append((i, j))
            else:
                disconcordant_pair.append((i, j))
        else:
            tie_uncomparable.append((i, j))

print("Total Pairs:", len(ref) * (len(ref) - 1) // 2)
print("Comparable Pairs:", len(concordant_pair) + len(disconcordant_pair) + len(tie_pair))
print("Concordant:", len(concordant_pair))
print("Discordant:", len(disconcordant_pair))
print("Pilot ties:", len(tie_pair))
print("Ref ties / uncomparable:", len(tie_uncomparable))
print("C-index:", (len(concordant_pair) + 0.5 * len(tie_pair)) /
      (len(concordant_pair) + len(disconcordant_pair) + len(tie_pair)))

Total Pairs: 1225
Comparable Pairs: 1104
Concordant: 736
Discordant: 205
Pilot ties: 163
Ref ties / uncomparable: 121
C-index: 0.7404891304347826


In [319]:
def calculate_concordance(ref, pil):
    concordant_count = 0.0
    total_comparable_pairs = 0
    n = len(ref)
    
    for i in range(n):
        for j in range(i + 1, n):
            # We only care about pairs where the reference values are not tied
            # If ref[i] == ref[j], the pair is usually excluded from C-index
            if ref[i] != ref[j]:
                total_comparable_pairs += 1
                
                # Determine the 'direction' of the relationship
                ref_direction = ref[i] > ref[j]
                pil_direction = pil[i] > pil[j]
                
                if pil[i] == pil[j]:
                    # The '0.5' rule for ties in the pilot/prediction
                    concordant_count += 0.5
                elif ref_direction == pil_direction:
                    # Both lists agree on which one is larger
                    concordant_count += 1.0
                else:
                    # Discordant pair
                    concordant_count += 0.0
                    
    if total_comparable_pairs == 0:
        return 0
        
    return concordant_count / total_comparable_pairs

# Example data
ref = aa['ref.time'].values
pil = aa['pilot.time'].values

score = calculate_concordance(ref, pil)
print(f"Concordance Score: {score:.4f}")

Concordance Score: 0.7438


In [306]:
import numpy as np
import pandas as pd

def round_away_from_zero(x):
    x = np.asarray(x, dtype=float)
    return (np.sign(x) * np.floor(np.abs(x) + 0.5)).astype(np.int64)

# aa already threshold-filtered, per your message
df = aa.copy()

ref = pd.to_numeric(df["ref.time"], errors="raise").to_numpy()
pred = pd.to_numeric(df["pilot.time"], errors="raise").to_numpy()

ref_i  = round_away_from_zero(ref)
pred_i = round_away_from_zero(pred)

n = len(ref_i)
print("n =", n)

# ---- denom EXACTLY as R does it (ordered pairs, truth != truth2) ----
dt = ref_i[:, None] - ref_i[None, :]
mask = (dt != 0)
denom = int(mask.sum())
print("denom (R-style) =", denom)

# ---- identity check (must match denom) ----
vals, counts = np.unique(ref_i, return_counts=True)
sum_sq = int(np.sum(counts * counts))
denom_formula = int(n*n - sum_sq)
print("denom_formula =", denom_formula, "  (must equal denom)")

# print tie structure (this is what drives denom)
top = sorted(zip(vals, counts), key=lambda x: -x[1])[:15]
print("\nTop ref.time_int counts:")
for v, c in top:
    print(f"{v:>6d}: {c}")

# ---- now concordance exactly like R on ints ----
dp = pred_i[:, None] - pred_i[None, :]
prod = (dt * dp)[mask]
conc = int(np.sum(prod > 0))
ties = int(np.sum(prod == 0))
cindex = (conc + 0.5 * ties) / denom if denom else np.nan
print(f"\nconc={conc} ties={ties} cindex={cindex:.6f}")

n = 50
denom (R-style) = 2166
denom_formula = 2166   (must equal denom)

Top ref.time_int counts:
    -1: 15
    85: 7
   113: 3
   115: 3
   160: 3
   233: 3
    84: 2
   112: 2
   116: 2
   166: 2
 -3843: 1
 -3838: 1
 -3818: 1
   -17: 1
    89: 1

conc=1462 ties=298 cindex=0.743767


In [269]:
from IPython.display import display

quantiles_cindex = pd.DataFrame({
    'concordance': [tbl['cindex'].median()],
    'mean': [tbl['cindex'].mean()],
    'c25': [tbl['cindex'].quantile(0.25)],
    'c75': [tbl['cindex'].quantile(0.75)]
})
#drop index for cleaner display
print("Concordance (Full):")
display(quantiles_cindex.style.hide(axis="index").format(precision=3))

print("Concordance (Known Pilot):")
tbl_known = per_file_table_like_r(known_df, threshold=0.1)
quantiles_cindex_known = pd.DataFrame({
    'concordance': [tbl_known['cindex'].median()],
    'mean': [tbl_known['cindex'].mean()],
    'c25': [tbl_known['cindex'].quantile(0.25)],
    'c75': [tbl_known['cindex'].quantile(0.75)]
})
display(quantiles_cindex_known.style.hide(axis="index").format(precision=3))

print("Concordance (Unknown Pilot):")
tbl_unknown = per_file_table_like_r(unknown_df, threshold=0.1)
quantiles_cindex_unknown = pd.DataFrame({
    'concordance': [tbl_unknown['cindex'].median()],
    'mean': [tbl_unknown['cindex'].mean()],
    'c25': [tbl_unknown['cindex'].quantile(0.25)],
    'c75': [tbl_unknown['cindex'].quantile(0.75)]
})
display(quantiles_cindex_unknown.style.hide(axis="index").format(precision=3))
print("Concordance (Unknown Pilot, known_pilot == 0):")
tbl_unknown_0 = per_file_table_like_r(unknown_df_0, threshold=0.1)
quantiles_cindex_unknown_0 = pd.DataFrame({
    'concordance': [tbl_unknown_0['cindex'].median()],
    'mean': [tbl_unknown_0['cindex'].mean()],
    'c25': [tbl_unknown_0['cindex'].quantile(0.25)],
    'c75': [tbl_unknown_0['cindex'].quantile(0.75)]
})
display(quantiles_cindex_unknown_0.style.hide(axis="index").format(precision=3))

Concordance (Full):


concordance,mean,c25,c75
0.721,0.726,0.673,0.800


Concordance (Known Pilot):


concordance,mean,c25,c75
0.720,0.727,0.680,0.800


Concordance (Unknown Pilot):


concordance,mean,c25,c75
0.500,0.617,0.500,0.650


Concordance (Unknown Pilot, known_pilot == 0):


concordance,mean,c25,c75
0.500,0.617,0.500,0.650


#### R Code Output

In [281]:
import pandas as pd

# Create the data manually, similar to the structure you've provided
data = {
    'common.bns': [
        '2023-03-03_11-5', '2023-03-04_11-1', '2023-03-06_08-1', '2023-03-07_12-5', '2023-03-16_09-2',
        '2023-03-18_11-4', '2023-03-27_13-1', '2023-03-28_09-3', '2023-03-30_10-0', '2023-03-31_10-3',
        '2023-04-03_13-4', '2023-04-04_14-1', '2023-04-05_08-4', '2023-04-07_10-1', '2023-04-07_16-4',
        '2023-12-11_13-5', '2023-12-11_16-5', '2024-01-09_10-3', '2024-01-10_10-5', '2024-01-12_11-1'
    ],
    'pilot.names': ['DSL33'] * 20,  # Assuming same name for all rows
    'data': [None] * 20,  # Placeholder for the <tibble> in each row
    'cindex': [0.680, 0.639, 0.802, 0.573, 0.675, 0.736, 0.799, 0.758, 0.731, 0.655,
               0.792, 0.717, 0.625, 0.526, 0.640, 0.833, 0.609, 0.616, 0.709, 0.831],
    'score_sum': [695, 516, 361, 1132, 1045, 972, 267, 785, 174, 957,
                  779, 1169, 1288, 656, 1431, 5, 229, 2221, 730, 103],
    'denom': [1022, 808, 450, 1974, 1548, 1320, 334, 1036, 238, 1462,
              984, 1630, 2060, 1248, 2236, 6, 376, 3604, 1030, 124],
    'concordant': [634, 442, 322, 676, 826, 852, 246, 702, 142, 736,
                   672, 1002, 876, 314, 1092, 4, 156, 1846, 586, 92],
    'ties': [122, 148, 78, 912, 438, 240, 42, 166, 64, 442,
             214, 334, 824, 684, 678, 2, 146, 750, 288, 22]
}

# Create a pandas DataFrame
df_R = pd.DataFrame(data)

# Show the DataFrame
df_R

,common.bns,pilot.names,data,cindex,score_sum,denom,concordant,ties
0,2023-03-03_11-5,DSL33,None,0.680,695,1022,634,122
1,2023-03-04_11-1,DSL33,None,0.639,516,808,442,148
2,2023-03-06_08-1,DSL33,None,0.802,361,450,322,78
3,2023-03-07_12-5,DSL33,None,0.573,1132,1974,676,912
4,2023-03-16_09-2,DSL33,None,0.675,1045,1548,826,438
5,2023-03-18_11-4,DSL33,None,0.736,972,1320,852,240
6,2023-03-27_13-1,DSL33,None,0.799,267,334,246,42
7,2023-03-28_09-3,DSL33,None,0.758,785,1036,702,166
8,2023-03-30_10-0,DSL33,None,0.731,174,238,142,64
9,2023-03-31_10-3,DSL33,None,0.655,957,1462,736,442


In [283]:
quantiles_cindex_R = pd.DataFrame({
    'concordance': [df_R['cindex'].median()],
    'c25': [df_R['cindex'].quantile(0.25)],
    'c75': [df_R['cindex'].quantile(0.75)]
})
#drop index for cleaner display
quantiles_cindex_R

,concordance,c25,c75
0,0.6945,0.6355,0.7665


### AULTC

In [16]:
"""
AULTC (Area Under Log Time Curve) calculation matching R implementation.

This calculates the area under the empirical CDF curve of log-transformed
time discrepancies between matched annotations.
"""

import pandas as pd
import numpy as np


def ecdf_auc2(field_values, upper_limit):
    """
    Calculate AUC of empirical CDF.
    
    Matches R's ecdf_auc2 function which:
    1. Adds upper bound to data
    2. Sorts values
    3. Caps at upper limit
    4. Computes empirical CDF areas as rectangles
    5. Normalizes by upper limit
    
    Args:
        field_values: Array of log-transformed time differences
        upper_limit: Upper bound for the curve (log scale)
        
    Returns:
        Normalized AUC value
    """
    # Combine data with upper bound
    df = pd.DataFrame({'field': list(field_values) + [upper_limit]})
    
    # Sort by field
    df = df.sort_values('field').reset_index(drop=True)
    
    # Cap values at upper
    df['field'] = df['field'].clip(upper=upper_limit)
    
    # Calculate next value (lead)
    df['field2'] = df['field'].shift(-1, fill_value=0)
    
    # Calculate empirical CDF positions
    # R: nth = (1:n())/(n()-1)
    n = len(df)
    df['nth'] = np.arange(1, n + 1) / (n - 1)
    
    # Calculate step size and box area
    df['dx'] = df['field2'] - df['field']
    df['box'] = df['dx'] * df['nth']
    
    # Remove last row (after computing boxes)
    df = df.iloc[:-1]
    
    # Sum boxes and normalize by upper limit
    auc = df['box'].sum() / upper_limit
    
    return float(auc)


def calculate_aultc(df, threshold=0.1, upper_limit=None, 
                    use_original_files=True,
                    man_has_header=True, pilot_has_header=True,
                    version_column=None):
    """
    Calculate AULTC (Area Under Log Time Curve) for time discrepancy.
    
    This matches R's calculate_time_discrepancy function.
    
    Args:
        df: best_matches DataFrame with columns:
            - error.rate: matching error rate
            - time/time.pilot OR ref.time/pilot.time: time values
            - optionally a version/grouping column
        threshold: Distance threshold for filtering matches
        upper_limit: Upper bound for log time (default: log(60*24*365.26) ≈ 1 year)
        use_original_files: If True, load times from original CSV files
        man_has_header: Whether manual CSVs have headers
        pilot_has_header: Whether pilot CSVs have headers  
        version_column: Column name to group by (e.g., 'pilot.names', 'Version')
                       If None, calculates overall AULTC
        
    Returns:
        If version_column is None: float (overall AULTC)
        If version_column is set: DataFrame with columns [version_column, 'auc']
    """
    if upper_limit is None:
        upper_limit = np.log(60 * 24 * 365.26)  # 1 year in log minutes
    
    out = df.copy()
    
    # Handle column naming - R uses 'time' and 'time.pilot' internally
    # but CSVs may have 'ref.time' and 'pilot.time'
    if use_original_files and 'files' in out.columns and 'pilot.files' in out.columns:
        # from concordance_calculator import load_times_from_files
        out = load_times_from_files(out, man_has_header, pilot_has_header)
        # Rename to match R's internal naming
        out = out.rename(columns={
            'ref.time_from_file': 'time',
            'pilot.time_from_file': 'time.pilot'
        })
    else:
        # Map CSV columns to R's internal naming
        if 'time' not in out.columns:
            if 'ref.time' in out.columns:
                out = out.rename(columns={'ref.time': 'time'})
            else:
                raise ValueError("Missing time column (need 'time' or 'ref.time')")
        
        if 'time.pilot' not in out.columns:
            if 'pilot.time' in out.columns:
                out = out.rename(columns={'pilot.time': 'time.pilot'})
            else:
                raise ValueError("Missing pilot time column (need 'time.pilot' or 'pilot.time')")
    
    # Apply R's filtering logic - EXACTLY matching R code line 520-524:
    # mutate(time.pilot = as.numeric(time.pilot)) %>%
    # filter(!is.na(time.pilot), !is.na(event)) %>%
    # mutate(keep=error.rate < threshold, threshold = threshold) %>%
    # filter(keep) %>%
    
    out['error.rate'] = pd.to_numeric(out['error.rate'], errors='coerce')
    out = out[(out['error.rate'].notna()) & 
              (out['error.rate'] < threshold) &
              (np.isfinite(out['error.rate']))].copy()
    
    if len(out) == 0:
        if version_column:
            return pd.DataFrame(columns=[version_column, 'auc'])
        return np.nan
    
    # Convert times to numeric - matching R exactly
    # R: as.numeric(time.pilot) and as.numeric(time)
    out['time'] = pd.to_numeric(out['time'], errors='coerce')
    out['time.pilot'] = pd.to_numeric(out['time.pilot'], errors='coerce')
    
    # R has additional filters!
    # filter(!is.na(time.pilot), !is.na(event))
    out = out[out['time.pilot'].notna()].copy()
    if 'event' in out.columns or 'v1' in out.columns:
        event_col = 'event' if 'event' in out.columns else 'v1'
        out = out[out[event_col].notna()].copy()
    
    if len(out) == 0:
        if version_column:
            return pd.DataFrame(columns=[version_column, 'auc'])
        return np.nan
    
    # Calculate absolute error
    # R: ae = abs(as.numeric(time.pilot) - as.numeric(time))
    out['ae'] = (out['time.pilot'] - out['time']).abs()
    
    # Calculate log absolute error + 1
    # R: lae = ifelse(is.na(ae), exp(upper_limit)+1, ae+1)
    out['lae'] = np.where(
        out['ae'].isna(),
        np.exp(upper_limit) + 1,
        out['ae'] + 1
    )
    
    # Group by version if specified, otherwise calculate overall
    if version_column and version_column in out.columns:
        # R: select(Version, lae) %>% nest(data=-Version)
        results = []
        for version, group in out.groupby(version_column, sort=False):
            if len(group) > 0:
                # R: field=lae %>% mutate(field=log(field)) %>% ecdf_auc2
                field_values = np.log(group['lae'].values)
                auc = ecdf_auc2(field_values, upper_limit)
                results.append({version_column: version, 'auc': auc})
        
        return pd.DataFrame(results)
    else:
        # Calculate overall AULTC
        field_values = np.log(out['lae'].values)
        return ecdf_auc2(field_values, upper_limit)


def calculate_aultc_simple(df, threshold=0.1, upper_limit=None):
    """
    Simplified AULTC calculation for flat best_matches DataFrame.
    
    Handles both column naming conventions:
    - R internal: 'time' and 'time.pilot'
    - CSV format: 'ref.time' and 'pilot.time'
    
    Args:
        df: DataFrame with error.rate and time columns
        threshold: Error rate threshold
        upper_limit: Upper bound for log time (default: log(1 year in minutes))
        
    Returns:
        Float AULTC value
    """
    if upper_limit is None:
        upper_limit = np.log(60 * 24 * 365.26)
    
    return calculate_aultc(
        df, 
        threshold=threshold,
        upper_limit=upper_limit,
        use_original_files=True,
        version_column=None
    )

In [134]:
result_aultc = calculate_aultc_simple(df, threshold=0.1, upper_limit=np.log(60*24*365.26))

print(f"(Full) {pilot_}, AULTC: {result_aultc:.3f}")
result_aultc_known = calculate_aultc_simple(known_df, threshold=0.1, upper_limit=np.log(60*24*365.26))
print(f"(Known Pilot) {pilot_}, AULTC: {result_aultc_known:.3f}")
result_aultc_unknown = calculate_aultc_simple(unknown_df, threshold=0.1, upper_limit=np.log(60*24*365.26))
print(f"(Unknown Pilot) {pilot_}, AULTC: {result_aultc_unknown:.3f}")
result_aultc_unknown_0 = calculate_aultc_simple(unknown_df_0, threshold=0.1, upper_limit=np.log(60*24*365.26))
print(f"(Unknown Pilot, known_pilot == 0) {pilot_}, AULTC: {result_aultc_unknown_0:.3f}")

(Full) DSL33, AULTC: 0.795
(Known Pilot) DSL33, AULTC: 0.797
(Unknown Pilot) DSL33, AULTC: 0.673
(Unknown Pilot, known_pilot == 0) DSL33, AULTC: 0.673


# Combined Metrics

In [197]:
df_combined = pd.DataFrame({
    'Metric': ['Match Rate', 'Concordance (Median)', 'Concordance (Mean)', 'C25', 'C75', 'AULTC'],
    'Full Pilot': [match_rate] + quantiles_cindex.iloc[0].values.tolist() + [result_aultc],
    'Known Pilot (known_pilot == 1)': [match_rate_known] + quantiles_cindex_known.iloc[0].values.tolist() + [result_aultc_known],
    'Unknown Pilot (known_pilot != 1)': [match_rate_unknown] + quantiles_cindex_unknown.iloc[0].values.tolist() + [result_aultc_unknown],
    'Unknown Pilot (known_pilot == 0)': [match_rate_unknown_0] + quantiles_cindex_unknown_0.iloc[0].values.tolist() + [result_aultc_unknown_0]
})

# Set 'Metric' column as index, then transpose
df_combined.set_index('Metric', inplace=True)

print(f"Full Pilot Matches:" f"{len(df)}")
print(f"Known Pilot Matches: {len(known_df)}")
print(f"Unknown Pilot Matches: {len(unknown_df)}")
print(f"Unknown Pilot Matches (known_pilot == 0): {len(unknown_df_0)}")

# Now, transpose and apply formatting
display(df_combined.transpose().style.format(precision=3))

Full Pilot Matches:3784
Known Pilot Matches: 1920
Unknown Pilot Matches: 1864
Unknown Pilot Matches (known_pilot == 0): 37


Metric,Match Rate,Concordance (Median),Concordance (Mean),C25,C75,AULTC
Full Pilot,0.372,0.721,0.726,0.673,0.800,0.795
Known Pilot (known_pilot == 1),0.722,0.720,0.727,0.680,0.800,0.797
Unknown Pilot (known_pilot != 1),0.011,0.500,0.617,0.500,0.650,0.673
Unknown Pilot (known_pilot == 0),0.541,0.500,0.617,0.500,0.650,0.673
